# Lab 1: MNIST with Classical Machine Learning
AI Lab, University of Technology Bahrain. About 10 minutes. CPU.

**You will learn**
- What the MNIST dataset is: 70,000 28×28 handwritten digits
- Flattening images into feature vectors
- Baselines: k-Nearest Neighbours and Logistic Regression
- Reading a confusion matrix

**Before you run**
- On Kaggle: right-hand panel, *Settings → Accelerator → GPU T4 x2* (when the lab needs a GPU) and *Settings → Internet → On*. GPU and Internet need a phone-verified account.
- On Colab: *Runtime → Change runtime type → T4 GPU*.
- Run the cells from top to bottom with **Shift + Enter**.

In [ ]:
# Setup: run this cell first
import os, sys, glob, time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

ON_KAGGLE = os.path.exists("/kaggle/working")
ON_COLAB = "google.colab" in sys.modules
OUT_DIR = "/kaggle/working" if ON_KAGGLE else "/content" if ON_COLAB else "."
gpus = tf.config.list_physical_devices("GPU")
print("Platform:", "Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "local")
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPU:", [g.name for g in gpus] if gpus else "none, running on CPU. See 'Before you run' above to switch a GPU on")
keras.utils.set_random_seed(42)

In [ ]:
# Dataset loader: downloads with Keras; on Kaggle it can also use a copy attached with "+ Add Input"
def load_dataset(name):
    """name: 'mnist', 'fashion_mnist' or 'cifar10' -> (x_train, y_train), (x_test, y_test)"""
    try:
        return getattr(keras.datasets, name).load_data()
    except Exception as err:
        if name == "mnist":  # Kaggle-attached copy of mnist.npz, if any
            hits = glob.glob("/kaggle/input/**/mnist.npz", recursive=True)
            if hits:
                d = np.load(hits[0])
                return (d["x_train"], d["y_train"]), (d["x_test"], d["y_test"])
        raise RuntimeError(
            "The dataset did not download. On Kaggle, switch on Settings > Internet "
            "(needs a phone-verified account). On Colab, check your connection. Then run this cell again.") from err

## 1. Look at the data

In [ ]:
(x_train, y_train), (x_test, y_test) = load_dataset("mnist")
print("train:", x_train.shape, "test:", x_test.shape, "pixel range:", x_train.min(), "-", x_train.max())

fig, axes = plt.subplots(2, 10, figsize=(12, 2.8))
for ax, i in zip(axes.flat, range(20)):
    ax.imshow(x_train[i], cmap="gray"); ax.set_title(y_train[i]); ax.axis("off")
plt.show()

## 2. Images → vectors
Classical models need a flat row of numbers per sample: 28 × 28 = **784 features**, scaled to [0, 1].

In [ ]:
Xtr = x_train.reshape(len(x_train), -1).astype("float32") / 255.0
Xte = x_test.reshape(len(x_test), -1).astype("float32") / 255.0
# Use 10k training images so this runs in seconds on CPU
Xtr_s, ytr_s = Xtr[:10000], y_train[:10000]
print(Xtr_s.shape)

## 3. Two baseline models

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

for name, model in [("k-NN (k=3)", KNeighborsClassifier(3, n_jobs=-1)),
                    ("Logistic Regression", LogisticRegression(max_iter=300))]:
    t = time.time(); model.fit(Xtr_s, ytr_s)
    acc = model.score(Xte[:2000], y_test[:2000])
    print(f"{name:22s} accuracy = {acc:.3f}   ({time.time()-t:.1f}s)")

## 4. Where does the model get confused?

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_estimator(model, Xte[:2000], y_test[:2000], cmap="Blues", colorbar=False)
plt.title("Logistic Regression: confusion matrix"); plt.show()

In [ ]:
# What did logistic regression learn? Each class has 784 weights: show them as images
fig, axes = plt.subplots(1, 10, figsize=(13, 1.8))
for d, ax in enumerate(axes):
    ax.imshow(model.coef_[d].reshape(28, 28), cmap="RdBu"); ax.set_title(d); ax.axis("off")
plt.suptitle("Red = pixel votes against, blue = votes for"); plt.show()

## Try it
1. Change `k` in k-NN to 1, 5 and 15. What happens to accuracy?
2. Train on all 60,000 images. How much better is it, and how much slower?
3. Which two digits are confused most often? Why might that be?

### Save your work
- **Kaggle:** *Save Version → Quick Save* keeps your edits; *Save & Run All* also keeps the outputs. Files written to `/kaggle/working` appear under **Output**.
- **Colab:** *File → Save a copy in Drive*. Files in `/content` are deleted when the session ends, so download anything you need from the Files panel first.